# Notebook 05 — Multi-Version YOLO Comparison (Part V)

**Objective:** Compare YOLO11 with at least THREE other YOLO versions on VisDrone under identical training conditions.

## Models Compared

| Model | Version | Params | Key Innovation |
|-------|---------|--------|----------------|
| YOLOv5su | v5 | ~7.2M | Anchor-based, first widely-adopted YOLO |
| YOLOv8s  | v8 | ~11.2M | Anchor-free, strong baseline |
| YOLOv9c  | v9 | ~25.3M | GELAN + PGI |
| YOLOv10s | v10 | ~7.2M | NMS-free (one-to-one head) |
| YOLO11s  | v11 | ~9.4M | Improved backbone + neck |
| YOLO26s  | v26 | ~9.5M | NMS-free, MuSGD optimizer, ProgLoss |

All models are trained for the same number of epochs at 640px on the same VisDrone split.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from ultralytics import YOLO

sns.set_theme(style='whitegrid')

DATA_CFG    = ROOT / 'configs' / 'visdrone.yaml'
PROJECT_DIR = ROOT / 'results' / 'comparison'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE      = '0'   # change to 'cpu' if needed

print('DATA_CFG:', DATA_CFG)
print('OUTPUT:  ', PROJECT_DIR)

## Shared Training Protocol

To ensure a fair comparison, **all models use identical hyperparameters**.

In [ ]:
SHARED_TRAIN_KWARGS = dict(
    data     = str(DATA_CFG),
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    lr0      = 0.01,
    lrf      = 0.01,
    weight_decay = 0.0005,
    patience = 20,
    device   = DEVICE,
    workers  = 4,
    project  = str(PROJECT_DIR),
    exist_ok = True,
    plots    = True,
    verbose  = False,
)

# Models to compare
MODELS = [
    ('yolov5su.pt',   'YOLOv5s'),
    ('yolov8s.pt',    'YOLOv8s'),
    ('yolov9c.pt',    'YOLOv9c'),
    ('yolov10s.pt',   'YOLOv10s'),
    ('yolo11s.pt',    'YOLO11s'),
    ('yolo26s.pt',    'YOLO26s'),
]

print(f'Training {len(MODELS)} models:')
for ckpt, name in MODELS:
    print(f'  {name:<12} ({ckpt})')

## Train All Models

In [ ]:
comparison_results = []

for checkpoint, model_name in MODELS:
    exp_name = f'compare_{model_name.lower().replace(" ", "_")}'

    print(f"\n{'─'*55}")
    print(f"  Training: {model_name}  ({checkpoint})")
    print(f"{'─'*55}")

    try:
        model = YOLO(checkpoint)

        # Count parameters
        n_params = sum(p.numel() for p in model.model.parameters()) / 1e6

        t0 = time.time()
        results = model.train(name=exp_name, **SHARED_TRAIN_KWARGS)
        elapsed = time.time() - t0

        # Inference speed benchmark (on val set)
        val_metrics = model.val(
            data=str(DATA_CFG), split='val', imgsz=640,
            device=DEVICE, verbose=False
        )
        rd = val_metrics.results_dict
        speed = val_metrics.speed  # dict: preprocess, inference, postprocess (ms)

        row = {
            'Model':         model_name,
            'Checkpoint':    checkpoint,
            'Params (M)':    round(n_params, 1),
            'mAP50':         round(float(rd.get('metrics/mAP50(B)', 0)), 4),
            'mAP50-95':      round(float(rd.get('metrics/mAP50-95(B)', 0)), 4),
            'Precision':     round(float(rd.get('metrics/precision(B)', 0)), 4),
            'Recall':        round(float(rd.get('metrics/recall(B)', 0)), 4),
            'Train Time (min)': round(elapsed / 60, 1),
            'Inference (ms)': round(speed.get('inference', 0), 2),
            'save_dir':      str(results.save_dir),
        }
        comparison_results.append(row)

        with open(Path(results.save_dir) / 'comparison_row.json', 'w') as f:
            json.dump(row, f, indent=2)

        print(f"  → mAP50={row['mAP50']:.4f}  mAP95={row['mAP50-95']:.4f}  "
              f"params={row['Params (M)']}M  infer={row['Inference (ms)']}ms")

    except Exception as e:
        print(f'  ERROR training {model_name}: {e}')
        comparison_results.append({
            'Model': model_name, 'Checkpoint': checkpoint,
            'error': str(e)
        })

print(f"\n\nAll {len(MODELS)} models completed.")

## Structured Comparison Table

In [ ]:
comp_df = pd.DataFrame([r for r in comparison_results if 'error' not in r])

display_cols = ['Model', 'Params (M)', 'mAP50', 'mAP50-95', 'Precision',
                'Recall', 'Train Time (min)', 'Inference (ms)']
comp_display = comp_df[display_cols].set_index('Model')

# Highlight best per metric
print('\n' + '='*75)
print('  Multi-Version YOLO Comparison Table')
print('='*75)
print(comp_display.to_string())

# Save
comp_display.to_csv(PROJECT_DIR / 'comparison_table.csv')
with open(PROJECT_DIR / 'comparison_results.json', 'w') as f:
    json.dump(comparison_results, f, indent=2)

print(f'\nSaved to {PROJECT_DIR}')

# Best per metric
for col in ['mAP50', 'mAP50-95', 'Precision', 'Recall']:
    best = comp_display[col].idxmax()
    print(f'  Best {col:<15}: {best} ({comp_display.loc[best, col]:.4f})')

## Visualisation 1 — mAP Comparison

In [ ]:
models_sorted = comp_display.sort_values('mAP50', ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(models_sorted)))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# mAP50 horizontal bar
bars = axes[0].barh(models_sorted.index, models_sorted['mAP50'], color=colors)
axes[0].set_xlabel('mAP@50')
axes[0].set_title('mAP@50 — YOLO Version Comparison')
for bar, v in zip(bars, models_sorted['mAP50']):
    axes[0].text(v + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{v:.4f}', va='center', fontsize=9)

# mAP50-95 horizontal bar
bars = axes[1].barh(models_sorted.index, models_sorted['mAP50-95'], color=colors)
axes[1].set_xlabel('mAP@50-95')
axes[1].set_title('mAP@50-95 — YOLO Version Comparison')
for bar, v in zip(bars, models_sorted['mAP50-95']):
    axes[1].text(v + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Multi-Version YOLO — mAP Comparison on VisDrone', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'map_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualisation 2 — Speed vs. Accuracy Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.tab10.colors
for i, (idx, row) in enumerate(comp_display.iterrows()):
    ax.scatter(row['Inference (ms)'], row['mAP50'],
               s=row['Params (M)'] * 15,  # bubble size = model size
               color=colors[i % len(colors)],
               label=idx, zorder=5, alpha=0.8, edgecolors='black', linewidth=0.5)
    ax.annotate(idx,
                (row['Inference (ms)'], row['mAP50']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Inference latency (ms/img)', fontsize=11)
ax.set_ylabel('mAP@50', fontsize=11)
ax.set_title('Speed vs. Accuracy Trade-off\n(bubble size ∝ model parameters)', fontsize=12)

# Add legend for bubble size
for size in [5, 15, 30]:
    ax.scatter([], [], s=size*15, c='gray', alpha=0.5, label=f'{size}M params')
ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(PROJECT_DIR / 'speed_accuracy_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualisation 3 — Radar / Spider Chart

In [ ]:
from matplotlib.patches import FancyArrowPatch
import numpy as np

metrics_for_radar = ['mAP50', 'mAP50-95', 'Precision', 'Recall']
# Invert Inference (lower is better) — use 1/inference normalised
radar_df = comp_display[metrics_for_radar].copy()

# Normalise to [0, 1]
norm_df = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min() + 1e-9)

categories = metrics_for_radar
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors = plt.cm.tab10.colors
for i, (model_name, row) in enumerate(norm_df.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=colors[i % len(colors)])
    ax.fill(angles, values, alpha=0.05, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Multi-Metric Radar Chart\n(normalised 0-1)', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig(PROJECT_DIR / 'radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualisation 4 — Confusion Matrix (YOLO11s)

In [ ]:
# The confusion matrix image is saved by Ultralytics automatically during val()
# Try to display it from the YOLO11s run
import glob

# Find the yolo11s run in comparison results
yolo11_run = next(
    (r for r in comparison_results if 'YOLO11s' in r.get('Model', '') and 'save_dir' in r), None
)
if yolo11_run:
    cm_path = Path(yolo11_run['save_dir']) / 'confusion_matrix_normalized.png'
    if not cm_path.exists():
        cm_path = Path(yolo11_run['save_dir']) / 'confusion_matrix.png'
    if cm_path.exists():
        from PIL import Image
        img = Image.open(cm_path)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Confusion Matrix — YOLO11s on VisDrone Val', fontsize=12)
        plt.tight_layout()
        plt.savefig(PROJECT_DIR / 'confusion_matrix_yolo11s.png', dpi=120, bbox_inches='tight')
        plt.show()
    else:
        print(f'Confusion matrix not found at {cm_path}')
else:
    print('YOLO11s run not found in comparison results')

## Loss Curve Comparison Across All Versions

In [ ]:
from scipy.ndimage import uniform_filter1d

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.tab10.colors

for i, row in enumerate([r for r in comparison_results if 'save_dir' in r]):
    csv_path = Path(row['save_dir']) / 'results.csv'
    if not csv_path.exists():
        continue
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    val_box = next((c for c in ['val/box_loss', 'val/box_om'] if c in df.columns), None)
    map50   = next((c for c in ['metrics/mAP50(B)', 'metrics/mAP_0.5'] if c in df.columns), None)

    color = colors[i % len(colors)]
    if val_box:
        s = uniform_filter1d(df[val_box].values, size=5)
        axes[0].plot(s, label=row['Model'], color=color, linewidth=2)
    if map50:
        axes[1].plot(df[map50].values, label=row['Model'], color=color, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Box Loss')
axes[0].set_title('Validation Box Loss — All Models')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP50')
axes[1].set_title('mAP50 Training Curve — All Models')
axes[1].legend(fontsize=8)

plt.suptitle('Training Dynamics: Multi-Version Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'training_curves_all_versions.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Summary Table

In [ ]:
# Rank models by mAP50
ranked = comp_display.sort_values('mAP50', ascending=False).copy()
ranked.insert(0, 'Rank', range(1, len(ranked)+1))

print('\nFinal Comparison Table (ranked by mAP50)')
print('=' * 80)
print(ranked.to_string())

ranked.to_csv(PROJECT_DIR / 'final_comparison_ranked.csv')
print(f'\nSaved to {PROJECT_DIR / "final_comparison_ranked.csv"}')

## Discussion

### Architecture Evolution

| Generation | Key Change | Impact on VisDrone |
|---|---|---|
| YOLOv5 | Anchor-based, CSPNet backbone | Baseline detection capability |
| YOLOv8 | Anchor-free, decoupled head | Better precision for small objects |
| YOLOv9 | GELAN + PGI (Programmable Gradient Info) | Better gradient flow, more accurate |
| YOLOv10 | NMS-free, one-to-one matching | Faster post-processing |
| YOLO11 | Improved backbone/neck, fewer params | Efficiency without accuracy loss |
| YOLO26 | NMS-free end-to-end, MuSGD, ProgLoss | Fastest CPU, best small-object loss |

### Key Findings
1. **Accuracy**: (fill in which model achieved best mAP on VisDrone)
2. **Speed**: (fill in which model is fastest for inference)
3. **Efficiency**: (best accuracy/parameter ratio)
4. **Small object**: YOLO26 specifically targets small-object improvements via ProgLoss+STAL

### Conclusion
For VisDrone's unique challenge (tiny, dense objects from drone footage):
- (fill in recommendation based on your results)
- Tradeoff: (accuracy vs speed for this specific use case)